# 03 - Extraccion de Color

Demo y evaluacion cualitativa del modulo de color. Comparacion de K=3,5,7.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('..')

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import yaml
from PIL import Image

from src.color.extractor import ColorExtractor

In [ ]:
with open('../config/pipeline_config.yaml') as f:
    config = yaml.safe_load(f)

extractor = ColorExtractor(config)

## Funcion de visualizacion de paleta

In [ ]:
def show_palette(image, result):
    palette = result['palette']
    n = len(palette)
    fig, axes = plt.subplots(1, 2, figsize=(10, 4), gridspec_kw={'width_ratios': [1, 2]})
    axes[0].imshow(image); axes[0].axis('off'); axes[0].set_title('Prenda')
    
    swatches = np.zeros((1, n, 3), dtype=np.uint8)
    labels = []
    for i, c in enumerate(palette):
        swatches[0, i] = c['rgb']
        labels.append(f"{c.get('name','')}\n{c['percentage']*100:.0f}%")
    axes[1].imshow(swatches, aspect='auto')
    axes[1].set_xticks(range(n)); axes[1].set_xticklabels(labels)
    axes[1].set_yticks([])
    axes[1].set_title(f"Paleta - patron: {result.get('pattern')}")
    plt.tight_layout(); plt.show()

## Demo sobre imagenes de muestra

In [ ]:
sample_dir = Path('../data/raw/demo')
images = sorted(sample_dir.glob('*.jpg'))[:20]
print(f'Imagenes: {len(images)}')

In [ ]:
for p in images[:10]:
    img = Image.open(p)
    result = extractor.extract(img)
    show_palette(img, result)

## Comparacion K=3 vs K=5 vs K=7

In [ ]:
for k in [3, 5, 7]:
    cfg_k = dict(config)
    cfg_k['color'] = dict(config['color'])
    cfg_k['color']['n_clusters'] = k
    ex_k = ColorExtractor(cfg_k)
    img = Image.open(images[0])
    r = ex_k.extract(img)
    print(f'K={k}:', [(c.get('name'), round(c['percentage'], 2)) for c in r['palette']])